# 현실의 데이터 09: 대출 상환 · 신용평가

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/09_credit_default.ipynb)

**분야**: 금융 · 핀테크 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 신용평가 (CSS) · 대출 심사 · 공정성 · 설명가능성

> 카드 고객 3만 명. 다음 달 연체할 사람을 미리 알 수 있을까? 성별 · 나이를 써도 될까?

**데이터**: UCI Default of Credit Card Clients (약 4MB). Yeh & Lien (2009). 대만 카드 고객 30,000명의 2005년 4~9월 기록: 한도 · 성별 · 학력 · 혼인 · 나이 · 6달 상환 상태 · 청구액 · 납부액 → 다음 달 연체 여부 (Hugging Face 의 CSV 사본).

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 카드 고객 한 명 |
| ② 입력 X | 한도 · 지난 6달 상환 상태 · 청구액 · 납부액 (+ 성별 · 나이 · 학력?) |
| ③ 출력 y | 다음 달 연체할까 |
| ④ 평가지표 | ROC-AUC · 재현율, 집단별 차이 |
| ⑤ 기준선 | 아무도 연체 안 한다 |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Yeh & Lien (2009) The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients, Expert Systems with Applications](https://doi.org/10.1016/j.eswa.2007.12.020)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
URL = "https://huggingface.co/datasets/Lancer73/uci-credit-card-default/resolve/main/UCI_Credit_Card.csv"
df = pd.read_csv(download(URL, "data/uci_credit_card.csv")).rename(columns={"default.payment.next.month": "default", "PAY_0": "PAY_1"})
print(df.shape, f"default rate {df['default'].mean():.1%}")
df.sample(5, random_state=0)

(30000, 25) default rate 22.1%


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
8225,8226,20000.0,1,1,2,33,1,2,2,2,...,18453.0,19755.0,19288.0,0.0,2260.0,0.0,1600.0,0.0,644.0,0
10794,10795,20000.0,2,2,2,35,0,0,2,0,...,19000.0,19000.0,20000.0,3400.0,0.0,1000.0,0.0,1000.0,0.0,0
9163,9164,230000.0,2,1,1,44,1,-1,-1,-1,...,933.0,0.0,0.0,949.0,2873.0,933.0,0.0,0.0,0.0,0
26591,26592,100000.0,1,2,1,42,0,0,0,0,...,17758.0,18774.0,20272.0,5000.0,2000.0,2000.0,2000.0,2000.0,2000.0,0
6631,6632,150000.0,1,1,2,29,-2,-2,-2,-2,...,6469.0,5138.0,7810.0,6989.0,833.0,6488.0,5153.0,7833.0,7130.0,0


## 열어 보기

`PAY_1` = 지난달 상환 상태 (−1 = 제때, 1 = 한 달 밀림, 2 = 두 달 …). 이게 가장 강한 단서일까? 성별 (1 = 남, 2 = 여) 마다 연체율은?

In [3]:
print(df.groupby("PAY_1")["default"].agg(["mean", "size"]).round(3).T)
print(df.groupby("SEX")["default"].mean().round(3).to_dict(), "(1 = male, 2 = female)")
print(df.groupby(pd.cut(df["AGE"], [20, 30, 40, 50, 80]), observed=True)["default"].mean().round(3).to_dict())

PAY_1        -2        -1          0         1         2        3       4  \
mean      0.132     0.168      0.128     0.339     0.691    0.758   0.684   
size   2759.000  5686.000  14737.000  3688.000  2667.000  322.000  76.000   

PAY_1     5       6      7       8  
mean    0.5   0.545  0.778   0.579  
size   26.0  11.000  9.000  19.000  
{1: 0.242, 2: 0.208} (1 = male, 2 = female)
{Interval(20, 30, closed='right'): 0.224, Interval(30, 40, closed='right'): 0.204, Interval(40, 50, closed='right'): 0.233, Interval(50, 80, closed='right'): 0.254}


## 문제 하나 골라 보기: 고객 → 다음 달 연체할까

같은 모델을 두 가지로: 성별 · 나이 · 혼인 · 학력을 **넣을 때와 뺄 때**. 점수가 얼마나 다를까? 그리고 뺀 모델이 남녀에게 각각 얼마나 '연체 위험' 을 주는지 비교한다.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sensitive = ["SEX", "AGE", "MARRIAGE", "EDUCATION"]
features = [c for c in df.columns if c not in ("ID", "default")]
tr, te = train_test_split(df, test_size=0.3, stratify=df["default"], random_state=0)
print(f"nobody defaults: accuracy {np.mean(te['default'] == 0):.1%}")
for name, cols in [("all features", features), ("without sex/age/marriage/education", [c for c in features if c not in sensitive])]:
    m = HistGradientBoostingClassifier(random_state=0, max_iter=200, learning_rate=0.05).fit(tr[cols], tr["default"])
    s = m.predict_proba(te[cols])[:, 1]
    flag = s > 0.5
    by_sex = {k: f"{flag[(te['SEX'] == k).values].mean():.1%}" for k in (1, 2)}
    print(f"{name:36s} ROC-AUC {roc_auc_score(te['default'], s):.3f}  recall {flag[te['default'].values == 1].mean():.1%}  flagged by sex {by_sex}")

nobody defaults: accuracy 77.9%


all features                         ROC-AUC 0.775  recall 35.8%  flagged by sex {1: '12.4%', 2: '11.1%'}


without sex/age/marriage/education   ROC-AUC 0.775  recall 36.1%  flagged by sex {1: '12.1%', 2: '11.3%'}


**관찰**: 성별 · 나이를 빼도 점수가 거의 줄지 않는다. 상환 기록이 훨씬 강한 단서이기 때문이다. 그래도 뺀 모델이 집단마다 다르게 '위험' 을 줄 수 있다 (다른 열이 성별을 대신 알려 줄 때). 실제 대출 심사에서는 법 · 규제로 쓸 수 없는 정보가 정해져 있고, 거절 이유를 설명할 수 있어야 한다.

## 다른 문제로 바꿔 보기

- **한도 정하기**: '연체할까' 대신 '얼마까지 빌려줄까' 는 회귀 문제 + 손익 계산이다.
- 거절당한 사람은 데이터에 없다 (승인된 사람만 연체 기록이 생긴다). 이 치우침을 어떻게 다룰까?

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |